In [2]:
import pandas as pd

In [3]:
import sys
print(sys.executable)

c:\Users\HP VICTUS\Documents\DIEGO\Programación\Portafolio\calidad-aire-bogota\.venv\Scripts\python.exe


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## Proyecto 1: Calidad del aire en Bogotá
**Pregunta:** ¿Qué estaciones y qué horas superan con más frecuencia el límite de PM2.5?
**Librerías:** pandas (cargar y transformar), matplotlib y seaborn (gráficos).
**Tiempo estimado:** 1 a 2 semanas.
**Por qué este proyecto:** trabajamos con datos reales, sucios, de una fuente que conozco por dentro.


In [12]:
ruta = "../data/raw/rmcab_pm25_2025.xlsx"
raw = pd.read_excel(ruta)

In [13]:
print(raw.shape)
raw.head(10)

(8772, 21)


,Multiestación: Periodica:01/01/2025 01:00-01/01/2026 00:00 Tipo:AVG 1 Hr.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fecha & Hora,Bolivia,Bosa,Carvajal - Sevillana,Centro de Alto Rendimiento,Ciudad Bolivar,Colina,Fontibon,Guaymaral,Jazmin,Kennedy,Las Ferias,MinAmbiente,Movil Fontibon,Movil 7ma,Puente Aranda,San Cristobal,Suba,Tunal,Usaquen,Usme
2,NaN,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5,PM2.5
3,NaN,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3,µg/m3
4,01/01/2025 01:00,57.1,Sin Data,160,12.4,74.8,41.1,106.6,18,25.5,66.5,74,10,27.3,9.4,15.4,31,35,42,0,65.6
5,01/01/2025 02:00,123.8,Sin Data,85,23.9,157.9,45.8,234.6,24,91.7,136.5,74,7,48.9,18.2,28.9,56,41,103,4,83.3
6,01/01/2025 03:00,146,Sin Data,120,47.1,172.2,47.9,105.4,31,86.4,177.8,146,10,71.8,13,60,41,96,119,7,94.1
7,01/01/2025 04:00,133.6,Sin Data,97,66.7,188.3,55.4,107.2,58,105.7,143.7,100,17,62,14.1,66.2,21,83,44,7,90.6
8,01/01/2025 05:00,116.5,Sin Data,85,82,185.2,61,79.6,44,99.9,126.1,64,19,65.2,23,75.4,19,66,95,9,74.7
9,01/01/2025 06:00,99.5,Sin Data,73,82.6,156.2,52.6,94.8,26,110.7,104.1,49,15,49.5,19.4,82.9,15,42,89,8,60.7


In [17]:
raw= pd.read_excel(ruta,skiprows=2)

In [18]:
raw.info()          

<class 'pandas.DataFrame'>
RangeIndex: 8770 entries, 0 to 8769
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Fecha & Hora                8768 non-null   str   
 1   Bolivia                     8770 non-null   object
 2   Bosa                        8770 non-null   object
 3   Carvajal - Sevillana        8770 non-null   object
 4   Centro de Alto Rendimiento  8770 non-null   object
 5   Ciudad Bolivar              8770 non-null   object
 6   Colina                      8770 non-null   object
 7   Fontibon                    8770 non-null   object
 8   Guaymaral                   8770 non-null   object
 9   Jazmin                      8770 non-null   object
 10  Kennedy                     8770 non-null   object
 11  Las Ferias                  8770 non-null   object
 12  MinAmbiente                 8770 non-null   object
 13  Movil Fontibon              8770 non-null   object
 14  Mov

In [19]:
raw.columns.tolist()

['Fecha & Hora',
 'Bolivia',
 'Bosa',
 'Carvajal - Sevillana',
 'Centro de Alto Rendimiento',
 'Ciudad Bolivar',
 'Colina',
 'Fontibon',
 'Guaymaral',
 'Jazmin',
 'Kennedy',
 'Las Ferias',
 'MinAmbiente',
 'Movil Fontibon',
 'Movil 7ma',
 'Puente Aranda',
 'San Cristobal',
 'Suba',
 'Tunal',
 'Usaquen',
 'Usme']

In [20]:
df = raw.rename(columns={"Fecha & Hora": "fecha_hora"})

In [21]:
#Algunos sistemas escriben la medianoche como 24:00, que pandas no entiende. Este código la convierte en las 00:00 del día siguiente:
if not pd.api.types.is_datetime64_any_dtype(df["fecha_hora"]):   # solo si la fecha vino como texto
    texto = df["fecha_hora"].astype(str).str.strip()
    es_24 = texto.str.contains("24:00")
    texto = texto.str.replace("24:00", "00:00", regex=False)
    df["fecha_hora"] = pd.to_datetime(texto, dayfirst=True, errors="coerce")
    df.loc[es_24, "fecha_hora"] += pd.Timedelta(days=1)

print(df["fecha_hora"].isna().sum(), "fechas que no se pudieron leer")


10 fechas que no se pudieron leer


**Convierte las mediciones a números.** Todo lo que no sea número (guiones, textos) queda como NaN:

In [22]:
estaciones = [c for c in df.columns if c != "fecha_hora"]
for col in estaciones:
    df[col] = pd.to_numeric(df[col], errors="coerce")


**Pasa a formato largo.** Hay una columna por estación (formato ancho). Para agrupar y graficar es más fácil tener tres columnas: fecha_hora, estación y valor. Eso hace melt:

In [24]:
largo = df.melt(id_vars="fecha_hora", var_name="estacion", value_name="pm25")
largo = largo.dropna(subset=["fecha_hora"])

In [25]:
# Valores negativos no tienen sentido físico: los marcamos como faltantes
largo.loc[largo["pm25"] < 0, "pm25"] = np.nan


In [26]:
largo.head()

,fecha_hora,estacion,pm25
2,2025-01-01 01:00:00,Bolivia,57.1
3,2025-01-01 02:00:00,Bolivia,123.8
4,2025-01-01 03:00:00,Bolivia,146.0
5,2025-01-01 04:00:00,Bolivia,133.6
6,2025-01-01 05:00:00,Bolivia,116.5


**Medir los datos faltantes por estación.** Este es un hallazgo en sí mismo:

In [30]:
faltantes = (largo.groupby("estacion")["pm25"]
             .apply(lambda s: s.isna().mean()*100)
             .sort_values(ascending=False)
             .round(1))

faltantes.head()

estacion
Bosa                    100.0
Colina                   29.0
Carvajal - Sevillana     18.6
Usaquen                  17.4
Tunal                    14.2
Name: pm25, dtype: float64

##Faltantes
Muchos faltantes se deben a que los equipos están en manteminiento o, hay una falla de internet que no pudo reportar el dato, o un corte de luz en las estaciones de monitoreo.

In [31]:
largo.to_csv("../data/pm25_horario_limpio.csv", index = False)